# CV MAE
Cross-validated MAE across folds.


In [1]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

MODEL_DIR = 'xgboost'
MODEL_NAME = 'XGBoost'

cwd = Path.cwd()
project_root = Path(".." ).resolve() if cwd.name == MODEL_DIR else cwd
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from qspr_config import (
    DATA_PATH,
    ECFP_RADIUS,
    ECFP_N_BITS,
    N_BITS_GRID,
    N_ESTIMATORS,
    N_ESTIMATORS_GRID,
    RANDOM_SEED,
    CV_FOLDS,
    N_JOBS,
    FIG_DPI,
    FIGSIZE_SQUARE,
)
from qspr_common import (
    load_dataset,
    build_feature_matrix,
    apply_plot_style,
    resolve_output_dir,
)

apply_plot_style()
out_dir = resolve_output_dir(MODEL_DIR)


In [2]:
df = load_dataset(DATA_PATH)
df, X = build_feature_matrix(df, radius=ECFP_RADIUS, n_bits=ECFP_N_BITS)
y = df["Solubility"].to_numpy()


[22:52:36] WARNING: not removing hydrogen atom without neighbors
[22:52:36] WARNING: not removing hydrogen atom without neighbors
[22:52:36] WARNING: not removing hydrogen atom without neighbors
[22:52:36] WARNING: not removing hydrogen atom without neighbors
[22:52:36] WARNING: not removing hydrogen atom without neighbors
[22:52:37] WARNING: not removing hydrogen atom without neighbors
[22:52:37] WARNING: not removing hydrogen atom without neighbors
[22:52:37] WARNING: not removing hydrogen atom without neighbors
[22:52:37] WARNING: not removing hydrogen atom without neighbors
[22:52:37] WARNING: not removing hydrogen atom without neighbors
[22:52:37] WARNING: not removing hydrogen atom without neighbors
[22:52:38] WARNING: not removing hydrogen atom without neighbors
[22:52:38] WARNING: not removing hydrogen atom without neighbors
[22:52:38] WARNING: not removing hydrogen atom without neighbors
[22:52:38] WARNING: not removing hydrogen atom without neighbors
[22:52:38] WARNING: not r

In [ ]:
from sklearn.model_selection import KFold
from xgboost import XGBRegressor

cv = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)

fold_mae = []
for train_idx, test_idx in cv.split(X):
    model = XGBRegressor(
        n_estimators=N_ESTIMATORS,
        random_state=RANDOM_SEED,
        n_jobs=N_JOBS,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="reg:squarederror",
        verbosity=0,
    )
    model.fit(X[train_idx], y[train_idx])
    pred = model.predict(X[test_idx])
    fold_mae.append(np.mean(np.abs(pred - y[test_idx])))

mean_mae = np.mean(fold_mae)
std_mae = np.std(fold_mae)

fig, ax = plt.subplots(figsize=FIGSIZE_SQUARE)
ax.bar(range(1, len(fold_mae) + 1), fold_mae, color="#4C78A8", alpha=0.85)
ax.axhline(mean_mae, color="black", linestyle="--", linewidth=1)
ax.set_xlabel("Fold")
ax.set_ylabel("MAE")
ax.set_title(f"{MODEL_NAME}: CV MAE (mean={mean_mae:.3f} +/- {std_mae:.3f})")
ax.set_xticks(range(1, len(fold_mae) + 1))
fig.tight_layout()

out_path = out_dir / "cv_mae.png"
fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight")
plt.show()
out_path
